> **Experiment D: Local per-turbine, no collaboration**
> Spec, decision rule and results: [docs/experiments/D_local_per_turbine.md](../docs/experiments/D_local_per_turbine.md)
> Depends on: E03 · Status: done · Config: [configs/arm_local_turbine.yaml](../configs/arm_local_turbine.yaml)
> Registered in `experiments/registry.yaml`; see the [experiment graph](../docs/experiments/EXPERIMENT_GRAPH.md).

# CARE to Compare · Wind Farm C · **Arm D: local per-turbine**

Arm A with one change: the unit is the turbine (`asset_id`, [ADR 0001](../docs/adr/0001-federated-client-is-asset-id.md)), not the event file. Each of the 22 models trains on all of its turbine's normal training rows, minus every row whose fingerprint appears in any of that turbine's prediction windows (the [E03](../docs/experiments/E03_client_dataset.md) rule, applied in memory). One scaler and one threshold per turbine; every event is scored by its turbine's model and CARE is computed over all 58 events as in A.

This is the no-collaboration end of the federated dial: every federated arm must beat it.

**Running:** on Kaggle the first cell clones the repo (internet on); locally set `WT_DATA_ROOT`. `WT_SEED` picks the seed (default 42). Output: `results/runs/D/<run id>/`.

In [1]:
# =============================================================================
# REPO + CONFIG  --  the run is described by configs/ and recorded in results/runs/
# =============================================================================
import os, sys, subprocess
from dataclasses import replace
from pathlib import Path

# Run from a checkout (this notebook inside notebooks/). On Kaggle the notebook
# stands alone, so the repo is cloned first (needs internet on).
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "experiments" / "registry.yaml").exists()), None)
if REPO is None:
    REPO = Path("/kaggle/working/Wind-Turbine")
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/PranavMundhra/Wind-Turbine.git",
                        str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))

from windturbine.config import load_config
from windturbine.paths import resolve_data_root
from windturbine.runs import (from_evaluate_care, new_run_dir, set_seeds, write_manifest,
                              write_results, write_scores)

EXP_ID = "D"
CFG = load_config(REPO / "configs" / "base.yaml", REPO / "configs" / "arm_local_turbine.yaml")
CFG = replace(CFG, seed=int(os.environ.get("WT_SEED", CFG.seed)))   # WT_SEED=43 etc. for the seed study
print(f"Experiment {EXP_ID} | repo {REPO} | seed {CFG.seed}")

Experiment D | repo c:\Users\pooja\Desktop\wind turbine\Wind-Turbine | seed 42


## Shared code
Identical to Arms A and B except two lines in data discovery and the cache: the data root comes from `resolve_data_root`, and `data/cache` in the repo is searched first.

In [2]:
# =============================================================================
# CONFIG  --  identical in both notebooks. Do not change one without the other.
# =============================================================================
import os, gc, re, json, math, time, hashlib, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# ---- scope -----------------------------------------------------------------
FARM               = "Wind Farm C"     # FAQ: status labels are only evaluation-
                                       # usable for Farms B and C. Farm A differs.

# ---- features (FAQ: "Avg features are often the safest starting point";
#      Min/Max/Std contain implausible values) ---------------------------------
USE_AVG_ONLY       = True
DROP_REACTIVE      = False             # set True to drop reactive_power_* sensors
ANGLE_TRANSFORM    = True              # FAQ lists angle transformation as a
                                       # typical preprocessing step
MAX_NAN_FRACTION   = 0.30              # drop a feature if >30% NaN in training

# ---- normal-behaviour filter (FAQ: filtering by status_type_id is intended
#      use; power-curve filtering is "also reasonable") -----------------------
NORMAL_STATUS_IDS      = {0, 2}
WIND_SPEED_CUTIN       = 3.0
WIND_SPEED_RATED       = 12.0
POWER_NEAR_ZERO_THRESH = 0.02

# ---- model (paper Table 3, Wind Farm C) ------------------------------------
HIDDEN_LAYERS      = [133, 83, 20, 83, 133]
LEARNING_RATE      = 0.0056            # paper Table 3
USE_LR_PLATEAU     = True
# At the paper's LR the TRAINING loss increases across epochs while validation
# plateaus -- the optimiser is overshooting, and early stopping then restores an
# under-trained model. ReduceLROnPlateau keeps the paper's starting LR but lets
# it decay when validation stalls. Set False to reproduce the paper exactly.
LR_PLATEAU_FACTOR  = 0.5
LR_PLATEAU_PATIENCE = 4
NOISE_STD          = 0.0               # no denoising noise for Farm C
BATCH_SIZE         = 64
MAX_EPOCHS         = 200
EARLY_STOP_PATIENCE = 10               # deviation from the 3 used earlier:
                                       # 3 stops on noise before convergence
VAL_FRACTION       = 0.15              # held out for early stopping AND for
                                       # threshold calibration (never trained on)
VAL_SPLIT_MODE     = "blocked"         # "blocked" | "chronological" | "random"
VAL_BLOCK_HOURS    = 120               # 5-day blocks for the blocked split
# RANDOM interleaves validation rows between training rows minutes apart, so
#   validation error measures within-period noise only. The threshold comes out
#   far too low for a prediction window months later, and the model over-flags.
# CHRONOLOGICAL takes the last VAL_FRACTION of the training period. Safe on
#   pooled data, but on a SINGLE event (~37k rows, ~8 months) that tail is one
#   contiguous season the model never saw, so validation error explodes and the
#   threshold with it -- observed spread of 42x across events.
# BLOCKED splits the period into VAL_BLOCK_HOURS blocks and takes every k-th one.
#   Validation rows are never adjacent to training rows (no autocorrelation
#   leakage) AND they span the full seasonal range (no regime shift). This is the
#   default for that reason.
SEED               = 42

# ---- thresholding ----------------------------------------------------------
# FAQ: "for early fault detection, you want to predict the event_label" -- NOT
# status_type_id. A normal-behaviour model has no anomaly labels at fit time, so
# the threshold is a high quantile of reconstruction error on held-out NORMAL
# data. Sweep SCORE_QUANTILE to trade Coverage against Accuracy.
SCORE_QUANTILE        = 0.99
CRITICALITY_THRESHOLD = 72             # FAQ: the documented default
BETA                  = 0.5
CARE_WEIGHTS          = (1, 1, 1, 2)   # (Coverage, Earliness, Reliability, Accuracy)

DTYPE = np.float32   # sensor dtype, used by every reader

np.random.seed(SEED)
print("Config loaded. Farm:", FARM)

Config loaded. Farm: Wind Farm C


In [3]:
# =============================================================================
# DATA DISCOVERY
# =============================================================================
DATASET_ROOT = str(resolve_data_root(cfg=CFG))   # WT_DATA_ROOT, configs/base.yaml or /kaggle/input

def find_farm_dir(root, farm_name):
    for dirpath, dirnames, filenames in os.walk(root):
        if os.path.basename(dirpath) == farm_name and "datasets" in dirnames:
            return dirpath
    raise FileNotFoundError(f"Could not locate '{farm_name}' under {root}")

FARM_DIR     = find_farm_dir(DATASET_ROOT, FARM)
DATASETS_DIR = os.path.join(FARM_DIR, "datasets")

def first_existing(*cands):
    for c in cands:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(cands)

event_info = pd.read_csv(first_existing(
    os.path.join(FARM_DIR, "comma_event_info.csv"),
    os.path.join(FARM_DIR, "event_info.csv")))
feature_description = pd.read_csv(first_existing(
    os.path.join(FARM_DIR, "comma_feature_description.csv"),
    os.path.join(FARM_DIR, "feature_description.csv")))

event_info["event_start"] = pd.to_datetime(event_info["event_start"], errors="coerce")
event_info["event_end"]   = pd.to_datetime(event_info["event_end"],   errors="coerce")

# The dataset ships each file twice (comma- and semicolon-delimited). Pick one
# variant so no event is loaded twice.
all_csvs = [f for f in os.listdir(DATASETS_DIR) if f.endswith(".csv")]
has_comma = any(f.startswith("comma_") for f in all_csvs)
has_plain = any(not f.startswith("comma_") for f in all_csvs)
if has_comma and has_plain:
    csv_files, DATA_SEP = [f for f in all_csvs if f.startswith("comma_")], ","
else:
    csv_files, DATA_SEP = all_csvs, ("," if has_comma else ";")

def extract_dataset_id(fname):
    m = re.search(r"\d+", fname)
    if not m:
        raise ValueError(f"No numeric id in {fname}")
    return int(m.group())

csv_files = sorted(csv_files, key=extract_dataset_id)
ids = [extract_dataset_id(f) for f in csv_files]
assert len(ids) == len(set(ids)), "Duplicate dataset ids after variant filtering"

n_anom   = (event_info["event_label"] == "anomaly").sum()
n_normal = (event_info["event_label"] == "normal").sum()
print(f"{len(csv_files)} event datasets | event_info: {n_anom} anomaly / {n_normal} normal")

# asset_id -> which physical turbine each event belongs to. Needed for the
# leakage diagnostics and for the pooled notebook's asset-disjoint split.
# Farm C's event_info has no asset_id column, but every DATA file carries one.
# Read a single cell from each file -- cheap, and without it we cannot tell
# which events share a turbine.
def build_event_to_asset():
    if "asset_id" in event_info.columns:
        return dict(zip(event_info["event_id"].astype(int),
                        event_info["asset_id"].astype(int)))
    m = {}
    for f in csv_files:
        try:
            v = pd.read_csv(os.path.join(DATASETS_DIR, f), sep=DATA_SEP,
                            usecols=["asset_id"], nrows=1)["asset_id"].iloc[0]
            m[extract_dataset_id(f)] = int(v)
        except Exception as e:
            print(f"  could not read asset_id from {f}: {e}")
    return m

EVENT_TO_ASSET = build_event_to_asset()
if EVENT_TO_ASSET:
    n_assets = len(set(EVENT_TO_ASSET.values()))
    print(f"{n_assets} distinct turbines across {len(csv_files)} events "
          f"({len(csv_files) - n_assets} events share a turbine with another event)")
else:
    print("WARNING: no asset_id anywhere -- effective-N and the pooled split "
          "will fall back to event-level handling")

58 event datasets | event_info: 27 anomaly / 31 normal
22 distinct turbines across 58 events (36 events share a turbine with another event)


In [4]:
# =============================================================================
# FEATURE SELECTION  --  Avg-only, optional angle transform
# =============================================================================
DESCRIPTIVE_COLS = ["time_stamp", "asset_id", "id", "train_test", "status_type_id"]

_sample = pd.read_csv(os.path.join(DATASETS_DIR, csv_files[0]), sep=DATA_SEP, nrows=200)
ALL_SENSOR_COLS = [c for c in _sample.columns if c not in DESCRIPTIVE_COLS]

if USE_AVG_ONLY:
    SENSOR_COLS = [c for c in ALL_SENSOR_COLS if c.endswith("_avg")]
else:
    SENSOR_COLS = list(ALL_SENSOR_COLS)

if DROP_REACTIVE:
    SENSOR_COLS = [c for c in SENSOR_COLS if not c.startswith("reactive_power_")]

print(f"{len(ALL_SENSOR_COLS)} sensor columns -> {len(SENSOR_COLS)} selected")

# ---- angle detection -------------------------------------------------------
# Prefer metadata; fall back to name prefixes. Always print what was found so
# the choice can be checked rather than trusted.
def detect_angle_sensors(feature_description, sensor_cols):
    angle_bases = set()
    # not `dtype == object`: pandas >= 3 stores text as "str", and that check found no angles
    text_cols = [c for c in feature_description.columns
                 if pd.api.types.is_string_dtype(feature_description[c]) and c != "sensor_name"]
    pat = re.compile(r"\b(degree|degrees|deg|angle|direction|azimuth)\b", re.I)
    for _, row in feature_description.iterrows():
        blob = " ".join(str(row[c]) for c in text_cols)
        if pat.search(blob):
            angle_bases.add(str(row["sensor_name"]))
    if not angle_bases:  # metadata gave nothing -> name-based fallback
        name_pat = re.compile(r"^(wind_direction|nacelle_direction|nacelle_position|"
                              r"pitch_angle|blade_angle|yaw_angle|wind_dir)", re.I)
        angle_bases = {n for n in feature_description["sensor_name"].astype(str)
                       if name_pat.match(n)}
    angle_cols = [c for c in sensor_cols
                  if any(c == f"{b}_avg" or c.startswith(f"{b}_") for b in angle_bases)]
    return sorted(set(angle_cols))

ANGLE_COLS = detect_angle_sensors(feature_description, SENSOR_COLS) if ANGLE_TRANSFORM else []
LINEAR_COLS = [c for c in SENSOR_COLS if c not in set(ANGLE_COLS)]
print(f"Angle columns detected ({len(ANGLE_COLS)}): {ANGLE_COLS if len(ANGLE_COLS) <= 12 else ANGLE_COLS[:12] + ['...']}")

# Sanity-check the detection: a true angle column spans roughly 0-360 (or
# -180..180). Anything outside that is a false positive and must NOT be sin/cos'd.
if ANGLE_COLS:
    _chk = pd.read_csv(os.path.join(DATASETS_DIR, csv_files[0]), sep=DATA_SEP,
                       usecols=ANGLE_COLS)
    _lo, _hi = _chk.min(), _chk.max()
    _suspect = [c for c in ANGLE_COLS
                if not ((-190 <= _lo[c] <= 5) and (170 <= _hi[c] <= 370))]
    print("\nAngle range check (first file):")
    for c in ANGLE_COLS:
        flag = "  <-- NOT angle-like" if c in _suspect else ""
        print(f"  {c:<22} [{_lo[c]:8.2f}, {_hi[c]:8.2f}]{flag}")
    if _suspect:
        print(f"\nDropping {len(_suspect)} false positives from the angle set.")
        ANGLE_COLS = [c for c in ANGLE_COLS if c not in _suspect]
        LINEAR_COLS = [c for c in SENSOR_COLS if c not in set(ANGLE_COLS)]
    del _chk

FEATURE_NAMES = LINEAR_COLS + [f"{c}__sin" for c in ANGLE_COLS] + [f"{c}__cos" for c in ANGLE_COLS]
N_FEATURES = len(FEATURE_NAMES)
print(f"Model input dimension: {N_FEATURES}")

PRESENT_DESCRIPTIVE = [c for c in DESCRIPTIVE_COLS if c in _sample.columns]
READ_COLS = PRESENT_DESCRIPTIVE + SENSOR_COLS
print(f"Columns actually parsed per file: {len(READ_COLS)} of {len(_sample.columns)}")

POWER_COL = next((f"{b}_avg" for b in feature_description["sensor_name"]
                  if str(b).startswith("power_") and f"{b}_avg" in SENSOR_COLS), None)
WIND_COL  = next((f"{b}_avg" for b in feature_description["sensor_name"]
                  if str(b).startswith("wind_speed_") and f"{b}_avg" in SENSOR_COLS), None)
print(f"Power-curve filter uses: power={POWER_COL}  wind={WIND_COL}")

952 sensor columns -> 238 selected
Angle columns detected (17): ['sensor_100_avg', 'sensor_101_avg', 'sensor_102_avg', 'sensor_103_avg', 'sensor_104_avg', 'sensor_105_avg', 'sensor_123_avg', 'sensor_124_avg', 'sensor_125_avg', 'sensor_126_avg', 'sensor_16_avg', 'sensor_49_avg', '...']

Angle range check (first file):
  sensor_100_avg         [   -2.00,    89.75]  <-- NOT angle-like
  sensor_101_avg         [   -2.00,    89.76]  <-- NOT angle-like
  sensor_102_avg         [   -2.00,    90.12]  <-- NOT angle-like
  sensor_103_avg         [   -1.95,    89.73]  <-- NOT angle-like
  sensor_104_avg         [   -1.91,    89.88]  <-- NOT angle-like
  sensor_105_avg         [   -1.98,    90.07]  <-- NOT angle-like
  sensor_123_avg         [ -166.52,   164.38]  <-- NOT angle-like
  sensor_124_avg         [ -165.38,   165.00]  <-- NOT angle-like
  sensor_125_avg         [ -269.90,   166.54]  <-- NOT angle-like
  sensor_126_avg         [ -269.90,   165.49]  <-- NOT angle-like
  sensor_16_avg      

In [5]:
# =============================================================================
# PARSED-DATA CACHE
# =============================================================================
# Each event CSV is ~55k rows x ~957 columns and takes 25-45s to parse. Arm A
# reads all 58 once; Arm B reads them twice. That is ~30-60 min of GPU-quota
# time during which the GPU is idle. Build the cache once in a CPU session
# (CPU sessions do not consume GPU quota), save /kaggle/working as a Kaggle
# Dataset, attach it to the GPU notebooks, and every later run skips parsing.

def resolve_cache_dir():
    """Prefer an attached cache dataset; fall back to the working dir."""
    for base in (str(REPO / CFG.cache_dir), "/kaggle/input", "/kaggle/working"):
        if not os.path.isdir(base):
            continue
        for dirpath, _, filenames in os.walk(base):
            if "care_cache_manifest.json" in filenames:
                return dirpath
    return os.path.join("/kaggle/working", "care_cache")

CACHE_DIR = resolve_cache_dir()
CACHE_MANIFEST = os.path.join(CACHE_DIR, "care_cache_manifest.json")
CACHE_READY = os.path.exists(CACHE_MANIFEST)

if CACHE_READY:
    _man = json.load(open(CACHE_MANIFEST))
    # A cache built for a different column set would silently change the model
    # input. Refuse it rather than train on the wrong features.
    if set(_man.get("sensor_cols", [])) != set(SENSOR_COLS):
        print(f"Cache at {CACHE_DIR} was built for a different feature set "
              f"({len(_man.get('sensor_cols', []))} cols vs {len(SENSOR_COLS)}). Ignoring it.")
        CACHE_READY = False
    else:
        print(f"Using cache: {CACHE_DIR}  ({_man['n_events']} events, "
              f"built {_man['built_at']})")
if not CACHE_READY:
    print(f"No usable cache. Parsing CSVs directly. "
          f"Run the cache-builder notebook to skip this next time.")

def cache_path(eid):
    return os.path.join(CACHE_DIR, f"event_{eid}.parquet")

def build_cache(out_dir=None, verbose=True):
    """Parse every event CSV once and write it as parquet. CPU session only."""
    import datetime
    out_dir = out_dir or os.path.join("/kaggle/working", "care_cache")
    os.makedirs(out_dir, exist_ok=True)
    t0 = time.time()
    for i, fname in enumerate(csv_files, 1):
        eid = extract_dataset_id(fname)
        df = pd.read_csv(os.path.join(DATASETS_DIR, fname), sep=DATA_SEP,
                         usecols=READ_COLS,
                         dtype={c: DTYPE for c in SENSOR_COLS})
        df["time_stamp"] = pd.to_datetime(df["time_stamp"])
        df = df.sort_values("time_stamp").reset_index(drop=True)
        df.to_parquet(os.path.join(out_dir, f"event_{eid}.parquet"),
                      index=False, compression="snappy")
        if verbose:
            print(f"[{i:>2}/{len(csv_files)}] event {eid:<3} {len(df):>6} rows "
                  f"({time.time()-t0:.0f}s elapsed)")
    json.dump({"sensor_cols": SENSOR_COLS, "descriptive_cols": PRESENT_DESCRIPTIVE,
               "n_events": len(csv_files), "farm": FARM, "sep": DATA_SEP,
               "built_at": datetime.datetime.now().isoformat(timespec="seconds")},
              open(os.path.join(out_dir, "care_cache_manifest.json"), "w"), indent=2)
    print(f"\nCache written to {out_dir} in {(time.time()-t0)/60:.1f} min")
    return out_dir

Using cache: c:\Users\pooja\Desktop\wind turbine\Wind-Turbine\data\cache  (58 events, built 2026-10-11T23:40:48)


In [6]:
# =============================================================================
# PREPROCESSING
# =============================================================================
def read_event_csv(fname):
    """Cache first, else parse only the columns actually used (~226 of 957)."""
    eid = extract_dataset_id(fname)
    if CACHE_READY and os.path.exists(cache_path(eid)):
        return pd.read_parquet(cache_path(eid))      # already sorted, typed
    df = pd.read_csv(os.path.join(DATASETS_DIR, fname), sep=DATA_SEP,
                     usecols=READ_COLS,
                     dtype={c: DTYPE for c in SENSOR_COLS})
    df["time_stamp"] = pd.to_datetime(df["time_stamp"])
    return df.sort_values("time_stamp").reset_index(drop=True)

def compute_normal_mask(df):
    """True = usable as normal-behaviour training data.

    FAQ: status_type_id filtering is the intended use; power-curve filtering is
    a reasonable addition. NOTE this is a TRAINING filter and also the mask for
    pointwise evaluation -- the FAQ excludes abnormal-status timestamps from
    pointwise scoring, because flagging a timestamp the operator already knew
    was abnormal earns no credit.
    """
    m = df["status_type_id"].isin(NORMAL_STATUS_IDS).to_numpy()
    if POWER_COL in df.columns and WIND_COL in df.columns:
        implausible = (df[WIND_COL].between(WIND_SPEED_CUTIN, WIND_SPEED_RATED)
                       & (df[POWER_COL].abs() <= POWER_NEAR_ZERO_THRESH)).to_numpy()
        m = m & ~implausible
    return m

def build_features(df):
    """Sensor frame -> model matrix. Angles become (sin, cos); NaNs stay NaN
    and are imputed AFTER scaling, so a missing value lands on the feature mean
    instead of becoming a large synthetic outlier."""
    parts = [df[LINEAR_COLS].to_numpy(dtype=np.float64)]
    if ANGLE_COLS:
        rad = np.deg2rad(df[ANGLE_COLS].to_numpy(dtype=np.float64))
        parts += [np.sin(rad), np.cos(rad)]
    return np.concatenate(parts, axis=1)

class Standardizer:
    """Fit on training rows only. Impute after scaling (0 == the mean)."""
    def __init__(self):
        self.mean_ = None; self.scale_ = None; self.keep_ = None

    def fit(self, X):
        with np.errstate(invalid="ignore"):
            nan_frac = np.isnan(X).mean(axis=0)
            self.keep_ = nan_frac <= MAX_NAN_FRACTION
            self.mean_ = np.nanmean(np.where(self.keep_, X, np.nan), axis=0)
            self.scale_ = np.nanstd(np.where(self.keep_, X, np.nan), axis=0)
        self.mean_ = np.nan_to_num(self.mean_, nan=0.0)
        self.scale_ = np.where(np.isfinite(self.scale_) & (self.scale_ > 1e-8),
                               self.scale_, 1.0)
        return self

    def transform(self, X):
        Z = (X - self.mean_) / self.scale_
        Z = np.nan_to_num(Z, nan=0.0, posinf=0.0, neginf=0.0)
        return Z[:, self.keep_].astype(DTYPE)

    @property
    def n_out(self):
        return int(self.keep_.sum())

def make_val_split(n, mode=None, frac=VAL_FRACTION, rng=None):
    """Return (train_idx, val_idx) over n time-ordered rows."""
    mode = mode or VAL_SPLIT_MODE
    cut = int(n * (1 - frac))
    if mode == "random":
        rng = rng or np.random.default_rng(SEED)
        idx = rng.permutation(n)
        return idx[:cut], idx[cut:]
    if mode == "chronological":
        return np.arange(cut), np.arange(cut, n)
    block = max(1, int(VAL_BLOCK_HOURS * 6))        # 6 rows per hour at 10-min
    stride = max(2, int(round(1.0 / frac)))         # every stride-th block -> val
    is_val = np.zeros(n, dtype=bool)
    for b in range(int(np.ceil(n / block))):
        if b % stride == stride - 1:
            is_val[b * block:(b + 1) * block] = True
    if is_val.sum() == 0 or is_val.sum() == n:      # too few rows for blocking
        return np.arange(cut), np.arange(cut, n)
    return np.flatnonzero(~is_val), np.flatnonzero(is_val)

def pick_threshold(val_scores, q=None, tag=""):
    """Quantile threshold plus a guard: a val block containing a regime the model
    never saw produces a handful of enormous residuals that drag the quantile up
    and silently disable detection for that event."""
    q = SCORE_QUANTILE if q is None else q
    thr = float(np.quantile(val_scores, q))
    med = float(np.median(val_scores))
    ratio = thr / max(med, 1e-9)
    if ratio > 10:
        print(f"    WARNING{' ' + tag if tag else ''}: threshold is {ratio:.0f}x the "
              f"median validation score (med={med:.2f}, thr={thr:.2f}). The "
              f"validation block likely contains an unseen operating regime.")
    return thr, dict(thr=thr, val_med=med, val_p99=float(np.quantile(val_scores, 0.99)),
                     val_max=float(val_scores.max()), thr_over_median=ratio)

def reconstruction_error(model, X, batch_size=8192):
    """L2 norm of the residual per row. Calls the model directly rather than
    .predict() -- in the per-event loop, .predict() retraces its tf.function on
    every new array shape and floods the log."""
    if len(X) == 0:
        return np.zeros(0, dtype=np.float32)
    out = np.empty(len(X), dtype=np.float32)
    for i in range(0, len(X), batch_size):
        chunk = X[i:i + batch_size]
        rec = model(chunk, training=False).numpy()
        out[i:i + len(chunk)] = np.linalg.norm(chunk - rec, axis=1)
    return out

In [7]:
# =============================================================================
# ROW FINGERPRINTS  --  leakage instrumentation
# =============================================================================
# The published dataset contains byte-identical rows across different event
# files (an acknowledged side effect of anonymization: "Overlaps across
# different event CSV files are an expected side effect"). A row that is
# `prediction` in one file can be `train` in another. Any notebook that pools
# training data across events therefore trains on its own evaluation rows.
# These fingerprints let each arm MEASURE that rather than assume it away.

def row_fingerprints(df, cols=None, decimals=6):
    cols = cols or SENSOR_COLS
    sub = df[cols].round(decimals)
    return pd.util.hash_pandas_object(sub, index=False).to_numpy()

def leakage_report(train_fp_set, pred_fp_by_event):
    rows = []
    for eid, fps in pred_fp_by_event.items():
        if len(fps) == 0:
            rows.append(dict(event_id=eid, pred_rows=0, leaked_rows=0, leaked_pct=0.0))
            continue
        hit = np.fromiter((f in train_fp_set for f in fps), dtype=bool, count=len(fps))
        rows.append(dict(event_id=eid, pred_rows=int(len(fps)),
                         leaked_rows=int(hit.sum()),
                         leaked_pct=round(100.0 * hit.mean(), 2)))
    rep = pd.DataFrame(rows).sort_values("leaked_pct", ascending=False)
    tot_p, tot_l = rep.pred_rows.sum(), rep.leaked_rows.sum()
    print(f"LEAKAGE: {tot_l:,} / {tot_p:,} prediction rows ({100*tot_l/max(tot_p,1):.2f}%) "
          f"were present in the training corpus")
    print(f"         {(rep.leaked_pct > 0).sum()} / {len(rep)} events affected; "
          f"{(rep.leaked_pct >= 99).sum()} events ~fully leaked")
    return rep

In [8]:
# =============================================================================
# MODEL  --  paper Table 3 architecture, Wind Farm C
# =============================================================================
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
tf.get_logger().setLevel("ERROR")
tf.random.set_seed(SEED)

def build_autoencoder(n_features, hidden=HIDDEN_LAYERS, noise=NOISE_STD, lr=LEARNING_RATE):
    inp = layers.Input(shape=(n_features,))
    x = layers.GaussianNoise(noise)(inp) if noise > 0 else inp
    for units in hidden:
        x = layers.Dense(units, activation="relu")(x)
    out = layers.Dense(n_features, activation="linear")(x)
    m = models.Model(inp, out)
    m.compile(optimizer=optimizers.Adam(learning_rate=lr), loss="mse")
    return m

def fit_autoencoder(Xtr, Xva, verbose=0, tag=""):
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    model = build_autoencoder(Xtr.shape[1])
    cbs = [callbacks.EarlyStopping(monitor="val_loss", patience=EARLY_STOP_PATIENCE,
                                   restore_best_weights=True, verbose=0)]
    if USE_LR_PLATEAU:
        cbs.append(callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=LR_PLATEAU_FACTOR,
            patience=LR_PLATEAU_PATIENCE, min_lr=1e-6, verbose=0))
    hist = model.fit(Xtr, Xtr, validation_data=(Xva, Xva),
                     batch_size=BATCH_SIZE, epochs=MAX_EPOCHS,
                     callbacks=cbs, verbose=verbose, shuffle=True)
    tr_hist = hist.history["loss"]
    info = dict(tag=tag, epochs_run=len(tr_hist),
                best_val=float(np.min(hist.history["val_loss"])),
                final_train=float(tr_hist[-1]),
                # >0 means training loss ended HIGHER than it started: divergence
                train_drift=float(tr_hist[-1] - min(tr_hist)))
    return model, info

In [9]:
# =============================================================================
# CARE SCORE  --  per the FAQ
# =============================================================================
# Coverage  : pointwise F-beta on anomalous events. Ground truth = timestamps
#             between event_start and event_end. Abnormal-status timestamps are
#             EXCLUDED from pointwise evaluation (Farm C rule).
# Accuracy  : true-negative rate on normal events, same normal-status mask.
# Earliness : weighted coverage inside the event window, early detections
#             weighted higher. Also restricted to normal-status timestamps so
#             it is measured on the same rows as Coverage.
# Reliability: ONE event-wise F-beta across all events (the FAQ notes it is not
#             averaged, because it is already computed across events).
#             Event decision = criticality counter reaching the threshold.

def fbeta(tp, fp, fn, beta=BETA):
    num = (1 + beta**2) * tp
    den = (1 + beta**2) * tp + (beta**2) * fn + fp
    return float(num / den) if den > 0 else 0.0

def event_ground_truth(times, start, end):
    if start is None or end is None or pd.isna(start) or pd.isna(end):
        return np.zeros(len(times), dtype=int)
    t = pd.to_datetime(times)
    return np.asarray((t >= start) & (t <= end)).astype(int)

def coverage_fbeta(rec, gt):
    m = rec["normal_mask"]; g = gt[m]; p = rec["prediction"][m]
    return fbeta(np.sum((p == 1) & (g == 1)),
                 np.sum((p == 1) & (g == 0)),
                 np.sum((p == 0) & (g == 1)))

def accuracy_care(rec):
    p = rec["prediction"][rec["normal_mask"]]
    tn, fp = np.sum(p == 0), np.sum(p == 1)
    return float(tn / (tn + fp)) if (tn + fp) > 0 else 1.0

def earliness_ws(rec, start, end):
    if start is None or end is None or pd.isna(start) or pd.isna(end):
        return None
    t = pd.to_datetime(rec["time_stamp"])
    inside = np.asarray((t >= start) & (t <= end)) & rec["normal_mask"]
    if inside.sum() == 0:
        return None
    span = (end - start).total_seconds()
    rel = (np.asarray(pd.to_datetime(rec["time_stamp"])[inside]) - np.datetime64(start))
    rel = rel.astype("timedelta64[s]").astype(float) / span if span > 0 else np.zeros(inside.sum())
    w = np.where(rel <= 0.5, 1.0, np.clip(1.0 - (rel - 0.5) / 0.5, 0.0, 1.0))
    return float((w * rec["prediction"][inside]).sum() / w.sum()) if w.sum() > 0 else 0.0

def criticality(status_normal, predictions):
    """FAQ: +1 when an anomaly is flagged on a normal-status timestamp, -1 when
    it is not, unchanged on abnormal-status timestamps. Floored at 0."""
    c = np.zeros(len(predictions) + 1, dtype=np.int32)
    for i in range(1, len(predictions) + 1):
        if not status_normal[i - 1]:
            c[i] = c[i - 1]
        elif predictions[i - 1] == 1:
            c[i] = c[i - 1] + 1
        else:
            c[i] = max(c[i - 1] - 1, 0)
    return c[1:]

def event_detected(rec, thr=CRITICALITY_THRESHOLD):
    c = criticality(rec["normal_mask"], rec["prediction"])
    return (int(c.max() >= thr), int(c.max()) if len(c) else 0) if len(c) else (0, 0)

def evaluate_care(meta, records, verbose=True):
    cov, ws, acc, per_event = [], [], [], []
    y_true, y_pred = [], []
    for eid, m in meta.items():
        rec = records[eid]
        gt = event_ground_truth(rec["time_stamp"], m["event_start"], m["event_end"])
        det, peak = event_detected(rec)
        row = dict(event_id=eid, asset_id=m.get("asset_id"),
                   is_anomaly=int(m["is_anomaly"]), detected=det, peak_criticality=peak,
                   n_pred_rows=int(len(rec["prediction"])),
                   flagged_pct=round(100 * float(rec["prediction"][rec["normal_mask"]].mean())
                                     if rec["normal_mask"].sum() else 0.0, 2))
        if m["is_anomaly"]:
            c = coverage_fbeta(rec, gt); cov.append(c); row["coverage"] = round(c, 4)
            w = earliness_ws(rec, m["event_start"], m["event_end"])
            if w is not None:
                ws.append(w); row["earliness"] = round(w, 4)
        else:
            a = accuracy_care(rec); acc.append(a); row["accuracy"] = round(a, 4)
        y_true.append(int(m["is_anomaly"])); y_pred.append(det)
        per_event.append(row)

    # A component with no supporting events is UNDEFINED, not zero. Treating it
    # as zero silently corrupts any subgroup analysis (e.g. a leaked-only slice
    # with no normal events would report Accuracy 0.0 and trip the Acc<0.5 rule).
    support = dict(coverage=len(cov), earliness=len(ws), accuracy=len(acc))
    C = float(np.mean(cov)) if cov else float("nan")
    E = float(np.mean(ws))  if ws  else float("nan")
    A = float(np.mean(acc)) if acc else float("nan")
    yt, yp = np.array(y_true), np.array(y_pred)
    tp = int(np.sum((yp == 1) & (yt == 1)))
    fp = int(np.sum((yp == 1) & (yt == 0)))
    fn = int(np.sum((yp == 0) & (yt == 1)))
    tn = int(np.sum((yp == 0) & (yt == 0)))
    R = fbeta(tp, fp, fn)

    complete = not (math.isnan(C) or math.isnan(E) or math.isnan(A))
    if not complete:
        care = float("nan")          # undefined on this subset
    elif yp.sum() == 0:
        care = 0.0                   # nothing flagged anywhere
    elif A < 0.5:
        care = A
    else:
        w1, w2, w3, w4 = CARE_WEIGHTS
        care = (w1*C + w2*E + w3*R + w4*A) / (w1 + w2 + w3 + w4)

    out = {"Coverage": C, "Earliness": E, "Reliability": R, "Accuracy": A,
           "CARE": float(care), "support": support,
           "event_confusion": {"tp": tp, "fp": fp, "fn": fn, "tn": tn},
           "n_anomaly_events": int(yt.sum()), "n_normal_events": int((yt == 0).sum())}
    if verbose:
        for k in ["Coverage", "Earliness", "Reliability", "Accuracy", "CARE"]:
            v = out[k]
            print(f"{k:<12} {'n/a  (no supporting events)' if math.isnan(v) else f'{v:.3f}'}")
        print(f"\nEvent-level confusion: tp={tp} fp={fp} fn={fn} tn={tn}")
        print(f"Support: {support['coverage']} anomaly events, "
              f"{support['accuracy']} normal events")
        if complete and care > 0:
            print(f"Accuracy term is {CARE_WEIGHTS[3]*A/(care*sum(CARE_WEIGHTS))*100:.0f}% "
                  f"of the CARE numerator")
        elif complete:
            print("CARE is 0 (no event flagged anywhere in this subset)")
    return out, pd.DataFrame(per_event)

In [10]:
# =============================================================================
# EFFECTIVE SAMPLE SIZE
# =============================================================================
# CARE averages over events as if they were independent. They are not: several
# events come from the same turbine and share rows outright. Report this next to
# the score so the number is read with the right error bars.
def effective_n_report(meta):
    assets = [m.get("asset_id") for m in meta.values() if m.get("asset_id") is not None]
    if not assets:
        print("No asset_id available; cannot report effective N.")
        return {}
    s = pd.Series(assets)
    counts = s.value_counts()
    print(f"{len(meta)} events span {counts.size} distinct turbines")
    print(f"  turbines contributing >1 event: {(counts > 1).sum()}")
    print(f"  largest single turbine: {counts.max()} events "
          f"({100*counts.max()/len(meta):.1f}% of the 'independent' evaluations)")
    return {"n_events": len(meta), "n_assets": int(counts.size),
            "max_events_per_asset": int(counts.max())}

## Run setup

In [11]:
# =============================================================================
# RUN SETUP  --  seed, run directory, manifest
# =============================================================================
# The cells above hard-code the arm constants (kept identical to Arms A and B).
# The manifest records CFG, so stop here if the two disagree.
_nb = dict(hidden_layers=tuple(HIDDEN_LAYERS), learning_rate=LEARNING_RATE,
           use_lr_plateau=USE_LR_PLATEAU, lr_plateau_factor=LR_PLATEAU_FACTOR,
           lr_plateau_patience=LR_PLATEAU_PATIENCE, noise_std=NOISE_STD, batch_size=BATCH_SIZE,
           max_epochs=MAX_EPOCHS, early_stop_patience=EARLY_STOP_PATIENCE, val_fraction=VAL_FRACTION,
           val_split_mode=VAL_SPLIT_MODE, val_block_hours=VAL_BLOCK_HOURS,
           score_quantile=SCORE_QUANTILE, criticality_threshold=CRITICALITY_THRESHOLD, beta=BETA,
           care_weights=tuple(CARE_WEIGHTS), use_avg_only=USE_AVG_ONLY, drop_reactive=DROP_REACTIVE,
           angle_transform=ANGLE_TRANSFORM, max_nan_fraction=MAX_NAN_FRACTION,
           normal_status_ids=tuple(sorted(NORMAL_STATUS_IDS)), wind_speed_cutin=WIND_SPEED_CUTIN,
           wind_speed_rated=WIND_SPEED_RATED, power_near_zero_thresh=POWER_NEAR_ZERO_THRESH)
_diff = {k: (v, getattr(CFG, k)) for k, v in _nb.items() if v != getattr(CFG, k)}
assert not _diff, f"notebook constants differ from configs/ (notebook, config): {_diff}"

SEED = CFG.seed                        # read by make_val_split, fit_autoencoder and the passes below
SEEDING = set_seeds(SEED)              # Python, NumPy, TensorFlow, plus op determinism
FP_DECIMALS = CFG.extra["fingerprint_decimals"]
RUN_DIR = new_run_dir(EXP_ID, CFG, root=REPO / CFG.output_dir)
write_manifest(RUN_DIR, EXP_ID, CFG, data_manifest=CACHE_MANIFEST if CACHE_READY else None,
               seeding=SEEDING, repo_root=REPO)
print("Run directory:", RUN_DIR)
print("Seeding:", SEEDING)

Run directory: c:\Users\pooja\Desktop\wind turbine\Wind-Turbine\results\runs\D\20261011T185853Z_2525b30e_s42
Seeding: {'seed': 42, 'tensorflow': True, 'op_determinism': True}


## Arm D

In [12]:
# =============================================================================
# ARM D -- LOCAL PER-TURBINE, NO COLLABORATION
# =============================================================================
ARM = CFG.arm
assert CFG.extra["client_unit"] == "asset_id" and CFG.extra["threshold_scope"] == "turbine"
assert CFG.extra["remove_rows_seen_in_prediction"]
print(ARM)

local_turbine


### Train one model per turbine

In [13]:
# =============================================================================
# TRAIN + SCORE, ONE TURBINE AT A TIME
# =============================================================================
rng = np.random.default_rng(SEED)
TURBINES = {}
for fname in csv_files:
    TURBINES.setdefault(EVENT_TO_ASSET[extract_dataset_id(fname)], []).append(fname)
print(f"{len(TURBINES)} turbines; {sum(len(v) == 1 for v in TURBINES.values())} have a single event")

dataset_meta, prediction_records, fit_log, client_rows = {}, {}, [], []
t0 = time.time()

for k, (aid, fnames) in enumerate(sorted(TURBINES.items()), 1):
    parts, pred = [], {}
    for fname in fnames:
        eid = extract_dataset_id(fname)
        df  = read_event_csv(fname)
        row = event_info[event_info["event_id"].astype(str) == str(eid)].iloc[0]
        is_anom = row["event_label"] == "anomaly"
        dataset_meta[eid] = dict(is_anomaly=bool(is_anom),
                                 event_start=row["event_start"] if is_anom else None,
                                 event_end=row["event_end"] if is_anom else None,
                                 asset_id=aid)

        tr_mask   = (df["train_test"] == "train").to_numpy()
        pr_mask   = (df["train_test"] == "prediction").to_numpy()
        norm_mask = compute_normal_mask(df)
        fp_all = row_fingerprints(df, decimals=FP_DECIMALS)
        X_all  = build_features(df)

        # A's blocked split, per event file (rows are time-ordered within a file)
        X_fit, fp_fit = X_all[tr_mask & norm_mask], fp_all[tr_mask & norm_mask]
        tr_idx, va_idx = make_val_split(len(X_fit), rng=rng)
        parts.append((X_fit[tr_idx], fp_fit[tr_idx], X_fit[va_idx], fp_fit[va_idx]))
        pred[eid] = dict(X=X_all[pr_mask], fp=fp_all[pr_mask],
                         time_stamp=df.loc[pr_mask, "time_stamp"].to_numpy(),
                         normal_mask=norm_mask[pr_mask])
        del df, X_all, X_fit, fp_fit, fp_all; gc.collect()

    X_tr, fp_tr, X_va, fp_va = (np.concatenate(p) for p in zip(*parts)); del parts
    n_tr0, n_va0 = len(X_tr), len(X_va)

    # ---- E03 rule: no training row that this turbine is evaluated on ----------
    pred_fp = np.concatenate([p["fp"] for p in pred.values()])
    tr_in_pred, va_in_pred = np.isin(fp_tr, pred_fp), np.isin(fp_va, pred_fp)
    X_va, fp_va = X_va[~va_in_pred], fp_va[~va_in_pred]
    # A turbine's files share rows (E02), so a row can sit in one file's training
    # blocks and in another file's validation blocks. Validation must be unseen,
    # so the copy goes from TRAINING: dropping it from validation instead can empty
    # the validation set of a turbine whose files are near-copies of each other.
    tr_in_va = ~tr_in_pred & np.isin(fp_tr, fp_va)
    X_tr, fp_tr = X_tr[~(tr_in_pred | tr_in_va)], fp_tr[~(tr_in_pred | tr_in_va)]
    assert len(X_va) and len(X_tr), f"turbine {aid}: empty train or validation set"

    # ---- fit on this turbine only ---------------------------------------------
    scaler = Standardizer().fit(X_tr)                  # fit on train slice ONLY
    Xtr, Xva = scaler.transform(X_tr), scaler.transform(X_va)
    model, info = fit_autoencoder(Xtr, Xva, tag=str(aid))

    va_scores = reconstruction_error(model, Xva)
    thr, thr_diag = pick_threshold(va_scores, tag=f"[turbine {aid}]")
    info.update(asset_id=aid, n_events=len(fnames), n_train=len(Xtr), n_val=len(Xva), **thr_diag)
    fit_log.append(info)
    client_rows.append(dict(asset_id=aid, n_events=len(fnames), events=" ".join(str(e) for e in pred),
                            train_rows_before=n_tr0, train_rows_removed_in_pred=int(tr_in_pred.sum()),
                            train_rows_removed_in_val=int(tr_in_va.sum()), val_rows_before=n_va0,
                            val_rows_removed_in_pred=int(va_in_pred.sum()),
                            train_rows=len(Xtr), val_rows=len(Xva)))

    # ---- score each of this turbine's events with its model -------------------
    for eid, p in pred.items():
        sc = reconstruction_error(model, scaler.transform(p["X"]))
        prediction_records[eid] = dict(time_stamp=p["time_stamp"], normal_mask=p["normal_mask"],
                                       scores=sc, prediction=(sc > thr).astype(int))

    print(f"[{k:>2}/{len(TURBINES)}] turbine {aid:<3} events={len(fnames)} "
          f"train={len(Xtr):>6} (-{n_tr0 - len(X_tr)}) val={len(Xva):>5} "
          f"epochs={info['epochs_run']:>3} val_mse={info['best_val']:.4f} thr={thr:.2f} "
          f"({(time.time()-t0)/60:.1f} min)")

    del X_tr, X_va, fp_tr, fp_va, Xtr, Xva, pred, model
    tf.keras.backend.clear_session(); gc.collect()

print(f"\nTrained {len(fit_log)} turbine models in {(time.time()-t0)/60:.1f} min")
fit_log_df = pd.DataFrame(fit_log)
client_manifest = pd.DataFrame(client_rows)
print(client_manifest.to_string(index=False))

22 turbines; 3 have a single event
[ 1/22] turbine 2   events=2 train= 67899 (-11786) val=12349 epochs= 29 val_mse=0.4674 thr=26.16 (1.1 min)
[ 2/22] turbine 5   events=1 train= 38689 (-0) val= 5760 epochs= 53 val_mse=0.0915 thr=17.09 (2.2 min)
[ 3/22] turbine 12  events=3 train= 84383 (-34087) val=17659 epochs= 35 val_mse=0.0928 thr=10.47 (4.0 min)
[ 4/22] turbine 13  events=1 train= 32929 (-0) val= 5040 epochs= 63 val_mse=0.0848 thr=9.81 (5.2 min)
[ 5/22] turbine 14  events=1 train= 36964 (-0) val= 5760 epochs= 46 val_mse=0.2256 thr=35.89 (6.2 min)
[ 6/22] turbine 15  events=2 train= 79457 (-0) val=12240 epochs= 35 val_mse=0.1344 thr=14.42 (7.5 min)
[ 7/22] turbine 16  events=4 train=142756 (-17384) val=25095 epochs= 27 val_mse=0.0885 thr=9.82 (9.5 min)
[ 8/22] turbine 21  events=2 train= 66458 (-11302) val=12049 epochs= 42 val_mse=0.1189 thr=15.23 (11.1 min)
[ 9/22] turbine 23  events=3 train=102291 (-10029) val=16947 epochs= 56 val_mse=0.0854 thr=10.92 (13.8 min)
[10/22] turbine 32

### CARE score

In [14]:
# =============================================================================
# RESULTS  ->  results/runs/D/<run id>/
# =============================================================================
results, per_event = evaluate_care(dataset_meta, prediction_records)
print()
results["effective_n"] = effective_n_report(dataset_meta)
results["arm"] = ARM
results["rows"] = {k: int(client_manifest[k].sum()) for k in
                   ["train_rows_before", "train_rows_removed_in_pred", "train_rows_removed_in_val",
                    "val_rows_before", "val_rows_removed_in_pred"]}

write_results(RUN_DIR, *from_evaluate_care(results, per_event))
write_scores(RUN_DIR, prediction_records)
fit_log_df.to_csv(RUN_DIR / "fit_log.csv", index=False)
client_manifest.to_csv(RUN_DIR / "client_manifest.csv", index=False)
print(f"\nSaved to {RUN_DIR}")
display(per_event)

Coverage     0.322
Earliness    0.231
Reliability  0.659
Accuracy     0.956
CARE         0.625

Event-level confusion: tp=12 fp=4 fn=15 tn=27
Support: 27 anomaly events, 31 normal events
Accuracy term is 61% of the CARE numerator

58 events span 22 distinct turbines
  turbines contributing >1 event: 19
  largest single turbine: 6 events (10.3% of the 'independent' evaluations)

Saved to c:\Users\pooja\Desktop\wind turbine\Wind-Turbine\results\runs\D\20261011T185853Z_2525b30e_s42


,event_id,asset_id,is_anomaly,detected,peak_criticality,n_pred_rows,flagged_pct,coverage,earliness,accuracy
0,12,2,1,0,1,3547,0.34,0.0163,0.0036,NaN
1,36,2,0,0,6,2888,2.36,NaN,NaN,0.9764
2,89,5,0,0,1,2737,0.27,NaN,NaN,0.9973
3,15,12,1,0,10,2737,1.11,0.0594,0.0123,NaN
4,50,12,0,0,64,3457,2.82,NaN,NaN,0.9718
5,66,12,1,1,127,1807,17.22,0.0000,0.0000,NaN
6,29,13,0,0,62,2305,10.44,NaN,NaN,0.8956
7,85,14,0,0,1,1585,0.13,NaN,NaN,0.9987
8,64,15,0,0,34,1873,4.73,NaN,NaN,0.9527
9,78,15,1,0,0,586,0.00,0.0000,0.0000,NaN


### Fit diagnostics and the single-event sanity check

In [15]:
# Single-event turbines get (almost) the data their Arm A model had: compare these
# rows with A's per-event results once A is re-run with a run directory.
single = client_manifest[client_manifest.n_events == 1].asset_id
display(per_event[per_event.asset_id.isin(single)])
display(fit_log_df.describe())
display(fit_log_df.sort_values("best_val", ascending=False).head(10))

,event_id,asset_id,is_anomaly,detected,peak_criticality,n_pred_rows,flagged_pct,coverage,earliness,accuracy
2,89,5,0,0,1,2737,0.27,NaN,NaN,0.9973
6,29,13,0,0,62,2305,10.44,NaN,NaN,0.8956
7,85,14,0,0,1,1585,0.13,NaN,NaN,0.9987


,epochs_run,best_val,final_train,train_drift,asset_id,n_events,n_train,n_val,thr,val_med,val_p99,val_max,thr_over_median
count,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000
mean,37.318182,0.151370,0.082133,0.001414,31.272727,2.636364,80640.681818,15157.136364,15.180066,4.730865,15.180066,55.917554,3.188300
std,20.272868,0.087381,0.060643,0.005109,17.091301,1.255292,30560.028430,6623.268329,6.978434,0.773160,6.978434,51.933252,1.276954
min,12.000000,0.073640,0.032673,0.000000,2.000000,1.000000,32929.000000,5040.000000,9.064833,3.581201,9.064833,17.609236,1.823344
25%,26.250000,0.095459,0.047015,0.000000,15.250000,2.000000,63788.250000,11520.000000,10.752500,4.153108,10.752500,29.624961,2.532848
50%,35.000000,0.121802,0.060003,0.000000,33.500000,2.000000,68796.000000,12529.500000,13.650531,4.730222,13.650531,36.546671,2.822023
75%,45.000000,0.169647,0.092794,0.000000,43.750000,3.750000,100554.250000,20482.250000,15.181331,5.082501,15.181331,63.957346,3.120609
max,104.000000,0.467392,0.245014,0.023689,56.000000,6.000000,142756.000000,29264.000000,35.886055,6.378468,35.886055,255.938354,7.342887


,tag,epochs_run,best_val,final_train,train_drift,asset_id,n_events,n_train,n_val,thr,val_med,val_p99,val_max,thr_over_median
0,2,29,0.467392,0.245014,0.023689,2,2,67899,12349,26.163136,6.378468,26.163136,255.938354,4.101790
13,38,26,0.261304,0.070263,0.000000,38,2,60916,11116,30.358761,5.114180,30.358761,111.561485,5.936194
4,14,46,0.225600,0.060060,0.000000,14,1,36964,5760,35.886055,4.887186,35.886055,75.327759,7.342887
19,53,21,0.217486,0.182009,0.005030,53,6,118857,29264,18.177040,6.019232,18.177040,102.319786,3.019827
15,43,12,0.206928,0.236630,0.002383,43,4,86340,22700,13.595585,6.133497,13.595585,59.231056,2.216612
17,50,30,0.172857,0.060043,0.000000,50,2,68855,12240,13.088725,4.850945,13.088725,65.532776,2.698180
21,56,31,0.160016,0.059963,0.000000,56,2,63687,12710,13.950701,5.340629,13.950701,28.659245,2.612183
10,33,39,0.149506,0.045382,0.000000,33,2,68737,10800,15.029880,4.987467,15.029880,34.244873,3.013530
5,15,35,0.134405,0.092343,0.000000,15,2,79457,12240,14.415204,4.893368,14.415204,29.730206,2.945866
18,52,40,0.129285,0.051912,0.000000,52,4,95344,21087,14.457706,4.609499,14.457706,38.743462,3.136502
